# 01 · Data collection

**Guideline covered:** Data collection

**What this notebook does.** Builds the dataset from scratch: Google Play Store reviews of the four largest UPI apps (BHIM, Google Pay, Paytm, PhonePe), collected with the open-source `google-play-scraper` library. No ready-made dataset was used.

**Two-stage collection.**
1. *Stage 1:* the newest 10,000 English reviews per app (India store), collected with [`src/scrape_reviews.py`](../src/scrape_reviews.py) and uploaded here (cell 4).
2. *Stage 2 (top-up, cell 1):* most Paytm and PhonePe reviews are one or two words long ("good app"), which say nothing about *what* went wrong. Scraping therefore continued until every app had at least **5,000 reviews of five or more words**. Short reviews were kept, so that rating statistics are not biased towards long, angry reviews.

**Result:** 86,056 reviews (BHIM 10,000 · Google Pay 18,221 · Paytm 27,614 · PhonePe 30,221), May to October 2026. Reviewer names were never stored.

**Inputs:** Google Play Store (public reviews) · **Outputs:** `data/raw/raw_reviews_v2.csv` (not published, see [data/README.md](../data/README.md))

**How to run:** open in Google Colab, keep the project folder at `MyDrive/upi-complaint-analytics` (or change `BASE` in the first code cell) and run the cells from top to bottom. The outputs shown are from the original run.

## Stage 2: top-up scrape until every app has 5,000 long reviews

In [ ]:
!pip install google-play-scraper

In [ ]:
!pip install -q google-play-scraper
import time, os, pandas as pd
from google_play_scraper import reviews, Sort
BASE = '/content/drive/MyDrive/upi-complaint-analytics'

TOPUP = {
    'PhonePe':    'com.phonepe.app',
    'Paytm':      'net.one97.paytm',
    'Google Pay': 'com.google.android.apps.nbu.paisa.user',
}
TARGET_LONG = 5000     # stop when this many reviews have 5+ words
MAX_TOTAL = 60000      # safety cap per app

def scrape(app_id, target_long=TARGET_LONG, cap=MAX_TOTAL):
    out, token, n_long = [], None, 0
    while n_long < target_long and len(out) < cap:
        batch, token = reviews(app_id, lang='en', country='in', sort=Sort.NEWEST,
                               count=200, continuation_token=token)
        if not batch:
            break
        out.extend(batch)
        n_long += sum(len(str(r['content'] or '').split()) >= 5 for r in batch)
        if len(out) % 2000 == 0:
            print(f'  {len(out):,} total, {n_long:,} long')
        if token is None:
            break
        time.sleep(1)
    return out

new = []
for name, app_id in TOPUP.items():
    print('Scraping', name)
    rows = scrape(app_id)
    for r in rows:
        r['app'] = name
    new.extend(rows)

new = pd.DataFrame(new)[['app', 'reviewId', 'content', 'score', 'thumbsUpCount',
                         'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt']]
new = new.rename(columns={'reviewId': 'review_id', 'content': 'review_text', 'score': 'rating',
                          'thumbsUpCount': 'helpful_votes', 'reviewCreatedVersion': 'app_version',
                          'at': 'review_date', 'replyContent': 'company_reply', 'repliedAt': 'reply_date'})

old = pd.read_csv(f'{BASE}/data/raw/raw_reviews.csv')
combined = pd.concat([old, new]).drop_duplicates(subset='review_id')
combined.to_csv(f'{BASE}/data/raw/raw_reviews_v2.csv', index=False, encoding='utf-8-sig')

check = pd.read_csv(f'{BASE}/data/raw/raw_reviews_v2.csv', parse_dates=['review_date'])
check['words'] = check['review_text'].fillna('').str.split().str.len()
print(check.shape)
print(check.groupby('app').agg(reviews=('review_id', 'count'),
                               first=('review_date', 'min'),
                               last=('review_date', 'max'),
                               usable_5plus_words=('words', lambda s: (s >= 5).sum())))

Scraping PhonePe
  2,000 total, 378 long
  4,000 total, 741 long
  6,000 total, 1,048 long
  8,000 total, 1,366 long
  10,000 total, 1,693 long
  12,000 total, 1,999 long
  14,000 total, 2,302 long
  16,000 total, 2,717 long
  18,000 total, 3,019 long
  20,000 total, 3,341 long
  22,000 total, 3,633 long
  24,000 total, 3,956 long
  26,000 total, 4,315 long
  28,000 total, 4,651 long
  30,000 total, 4,977 long
Scraping Paytm
  2,000 total, 251 long
  4,000 total, 516 long
  6,000 total, 844 long
  8,000 total, 1,115 long
  10,000 total, 1,391 long
  12,000 total, 1,725 long
  14,000 total, 2,226 long
  16,000 total, 2,676 long
  18,000 total, 3,096 long
  20,000 total, 3,574 long
  22,000 total, 3,967 long
  24,000 total, 4,350 long
  26,000 total, 4,735 long
Scraping Google Pay
  2,000 total, 610 long
  4,000 total, 1,126 long
  6,000 total, 1,469 long
  8,000 total, 1,905 long
  10,000 total, 2,495 long
  12,000 total, 3,074 long
  14,000 total, 3,677 long
  16,000 total, 4,307 long


Cell 3: connect Drive and create the project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/upi-complaint-analytics'
for sub in ['data/raw', 'data/processed', 'reports/figures', 'dashboard']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)

print('raw_reviews.csv still in Colab:', os.path.exists('/content/raw_reviews.csv'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
raw_reviews.csv still in Colab: False


## Upload the stage-1 file (scraped with `src/scrape_reviews.py`)

In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()   # a "Choose Files" button appears below the cell
for name in uploaded:
    shutil.move(name, f'{BASE}/data/raw/raw_reviews.csv')
print('In Drive:', os.listdir(f'{BASE}/data/raw'))

Saving raw_reviews.csv to raw_reviews.csv
In Drive: ['raw_reviews.csv']


Cell 4: copy all outputs to Drive, set the seed, check the data

In [ ]:
import shutil, random, numpy as np, pandas as pd

for f in ['raw_reviews.csv', 'dataset_summary.csv']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'{BASE}/data/raw/{f}')
for f in ['chart_rating_distribution.png', 'chart_review_length.png']:
    if os.path.exists(f'/content/{f}'):
        shutil.copy(f'/content/{f}', f'{BASE}/reports/figures/{f}')
print('Saved in Drive:', os.listdir(f'{BASE}/data/raw'))

random.seed(42); np.random.seed(42)

df = pd.read_csv(f'{BASE}/data/raw/raw_reviews.csv', parse_dates=['review_date'])
df['words'] = df['review_text'].fillna('').str.split().str.len()
print(df.shape)
print(df.groupby('app').agg(first=('review_date', 'min'),
                            last=('review_date', 'max'),
                            usable_5plus_words=('words', lambda s: (s >= 5).sum())))

Saved in Drive: ['raw_reviews.csv']
(40000, 10)
                         first                last  usable_5plus_words
app                                                                   
BHIM       2026-06-21 23:57:39 2026-10-14 17:19:26                7029
Google Pay 2026-07-28 06:41:56 2026-10-14 18:46:22                2493
Paytm      2026-09-17 05:27:03 2026-10-14 19:05:21                1456
PhonePe    2026-09-25 02:58:07 2026-10-14 18:43:38                1624
